## Phase 0: Environment Setup

In [4]:
import logging, os, sys, openpyxl, xlrd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler("pipeline.log"),
        logging.StreamHandler(sys.stdout)
    ]
)

In [6]:
print(os.getcwd())
logging.info("Current working directory: %s", os.getcwd())

d:\Abdulrhman\DEPI\Python Workspace\projects
2026-08-31 03:15:47,625 | INFO | Current working directory: d:\Abdulrhman\DEPI\Python Workspace\projects


# Phase 1: Data Loading & Initial Inspection

## 1.1 Import Data from Excel

In [ ]:
# 1.1 Import data from Excel
try:
    Orders = pd.read_excel(r"D:\Abdulrhman\DEPI\Datasets\Sample - Superstore 2019.xls", sheet_name="Orders")
    People = pd.read_excel(r"D:\Abdulrhman\DEPI\Datasets\Sample - Superstore 2019.xls", sheet_name="People")
    Returns = pd.read_excel(r"D:\Abdulrhman\DEPI\Datasets\Sample - Superstore 2019.xls", sheet_name="Returns")
    logging.info("Data imported successfully from Excel file: %s", r"D:\Abdulrhman\DEPI\Datasets\Sample - Superstore 2019.xls")
except Exception as e:
    logging.error("Failed to import data from Excel: %s", e)

## 1.2 Inspect Dataset Structure and Metadata

### 1.2.1 Inspect Orders Dataset

In [ ]:
# 1.2 Inspect dataset structure and metadata

logging.info("Starting dataset inspection")

# 1.2.1 Inspecting Orders dataset
print("Orders Shape:", Orders.shape)
print("Orders Columns:", Orders.columns.tolist())
print("Orders info:")
display(Orders.info(memory_usage="deep"))
print("\nOrders Sample:")
display(Orders.head(5))
print("\nOrders Random Sample:")
display(Orders.sample(5))

### 1.2.2 Inspect People Dataset

In [ ]:
logging.info("Starting dataset inspection")

# 1.2.2 Inspecting People dataset
print("People Shape:", People.shape)
print("People Columns:", People.columns.tolist())
print("People info:")
display(People.info())
print("\nPeople Sample:")
display(People.head())

### 1.2.3 Inspect Returns Dataset

In [ ]:
logging.info("Starting dataset inspection")

# 1.2.3 Inspecting Returns dataset
print("Returns Shape:", Returns.shape)
print("Returns Columns:", Returns.columns.tolist())
print("Returns info:")
display(Returns.info())
print("\nReturns Sample:")
display(Returns.head(5))
print("\nReturns Random Sample:")
display(Returns.sample(5))

# Phase 2: Raw Data Quality Assessment

## 2.1 Orders Data Quality Assessment

In [ ]:
# 2.1 Orders - Raw Data Quality Assessment

logging.info("Starting raw data quality Assessment")

# 2.1.1 Duplicate rows
print("Duplicate rows:", Orders.duplicated().sum())

# 2.1.2 Empty rows and columns
print("Empty rows:", Orders.isna().all(axis=1).sum())
print("Empty columns:", Orders.isna().all(axis=0).sum())

# 2.1.3 Missing values
print("\nMissing values:")
display(Orders.isna().sum())

# 2.1.4 Data types
print("\nData types:")
display(Orders.dtypes)

# 2.1.5 Valid values
print("\nInvalid values:")
print("Negative Sales:", (Orders["Sales"] < 0).sum())
print("Negative Quantity:", (Orders["Quantity"] < 0).sum())
print("Invalid Discount:", ((Orders["Discount"] < 0) | (Orders["Discount"] > 1)).sum())

# 2.1.6 Consistent categories
print("\nCategory values:")
print("Ship Mode:", Orders["Ship Mode"].unique())
print("Segment:", Orders["Segment"].unique())
print("Region:", Orders["Region"].unique())
print("Category:", Orders["Category"].unique())

# 2.1.7 Spaces
print("\nLeading/trailing or extra spaces:")

spaces_found = False
for col in Orders.select_dtypes(include="str").columns:
    spaces = Orders[col].astype(str).str.contains(
        r"^\s|\s$|\s{2,}",
        regex=True,
        na=False
    ).sum()
    if spaces > 0:
        print(f"{col}: {spaces}")
        spaces_found = True
if not spaces_found:
    print("no spaces found")

logging.info("Raw data quality checks completed successfully")

## 2.2 People Data Quality Assessment

In [ ]:
# 2.2 People - Raw Data Quality Assessment

logging.info("Starting People raw data quality assessment")

# 2.2.1 Duplicate rows
print("\n1. Duplicate rows:", People.duplicated().sum())

# 2.2.2 Empty rows and columns
print("Empty rows:", People.isna().all(axis=1).sum())
print("Empty columns:", People.isna().all(axis=0).sum())

# 2.2.3 Missing values
print("\nMissing values:")
display(People.isna().sum())

# 2.2.4 Data types
print("\nData types:")
display(People.dtypes)

# 2.2.5 Consistent categories
print("\nCategory values:")
for col in People.select_dtypes(include="str").columns:
    print(f"{col}:", People[col].unique())

# 2.2.6 Spaces
print("\nLeading/trailing or extra spaces:")

spaces_found = False

for col in People.select_dtypes(include="str").columns:
    spaces = People[col].str.contains(
        r"^\s|\s$|\s{2,}",
        na=False
    ).sum()

    if spaces > 0:
        print(f"{col}: {spaces}")
        spaces_found = True

if not spaces_found:
    print("no spaces found")
logging.info("People raw data quality assessment completed successfully")

## 2.3 Returns Data Quality Assessment

In [ ]:
# 2.3 Returns - Raw Data Quality Assessment

logging.info("Starting Returns raw data quality assessment")

# 2.3.1 Duplicate rows
print("\nDuplicate rows:", Returns.duplicated().sum())

# 2.3.2 Empty rows and columns
print("Empty rows:", Returns.isna().all(axis=1).sum())
print("Empty columns:", Returns.isna().all(axis=0).sum())

# 2.3.3 Missing values
print("\nMissing values:")
display(Returns.isna().sum())

# 2.3.4 Data types
print("\nData types:")
display(Returns.dtypes)

# 2.3.5 Spaces
print("\nLeading/trailing or extra spaces:")

spaces_found = False
for col in Returns.select_dtypes(include="str").columns:
    spaces = Returns[col].str.contains(
        r"^\s|\s$|\s{2,}",
        regex=True,
        na=False
    ).sum()
    if spaces > 0:
        print(f"{col}: {spaces}")
        spaces_found = True

if not spaces_found:
    print("no spaces found")
logging.info("Returns raw data quality assessment completed successfully")

# Phase 3: Data Cleaning & Preparation

## 3.1 Data Type Correction

In [ ]:
# 3.1 Data Type Correction
 
logging.info("Starting data type correction")

Orders["Postal Code"] = (Orders["Postal Code"].astype("Int64").astype("str"))

logging.info("Data type correction completed successfully")

In [ ]:
print("Data types:")
display(Orders.dtypes)


In [ ]:
Orders.head(5)

## 3.2 Missing Values Handling

In [ ]:
# 3.2 Missing Values handling

logging.info("Starting missing values assessment")

# Count missing values in each column

missing_values = Orders.isna().sum()
display(missing_values[missing_values > 0])

logging.info(f"Missing values found in {len(missing_values[missing_values > 0])} column(s)")

### 3.2.1 Identify Rows with Missing Postal Codes

In [ ]:
# Find rows with missing Postal Code & state & city

missing_postal = Orders[Orders["Postal Code"].isna()]
display(missing_postal[["City", "State", "Postal Code"]])

### 3.2.2 Validate Postal Code Information

In [ ]:
# Check Postal Code for Burlington, Vermont

burlington_postal = Orders[(Orders["City"] == "Burlington") & (Orders["State"] == "Vermont")]
display(burlington_postal[["City", "State", "Postal Code"]])

### 3.2.3 Fill Missing Postal Codes

In [ ]:
# Fill missing Postal Codes

Orders["Postal Code"] = Orders["Postal Code"].fillna("05401")

### 3.2.4 Validate Missing Postal Codes

In [ ]:
# Validate missing Postal Codes

missing = Orders.isna().sum()
display(missing)
logging.info(f"Remaining missing Postal Codes: {missing['Postal Code']}")

### 3.2.5 Create a Reusable Data Cleaning Class

In [ ]:
# Create a reusable data cleaning class

class DataCleaner:
    def __init__(self, df):
        self.df = df

    # Remove extra spaces from a column
    def remove_extra_spaces(self, column):
        self.df[column] = self.df[column].str.strip().str.replace(r"\s{2,}", " ", regex=True)

        spaces = self.df[column].str.contains(r"^\s|\s$|\s{2,}", regex=True, na=False).sum()

        if spaces == 0:
            logging.info(f"Extra spaces fixed in {column}")
        else:
            logging.warning(f"Extra spaces still found in {column}: {spaces}")

    # Remove duplicate rows
    def remove_duplicates(self):
        duplicates = self.df.duplicated().sum()

        if duplicates > 0:
            self.df = self.df.drop_duplicates(inplace=True)
            logging.info(f"Removed {duplicates} duplicate rows")
        else:
            logging.info("No duplicate rows found")
    

## 3.3 Duplicate Handling

In [ ]:
# 3.3 Duplicate Handling

logging.info("Starting duplicate assessment")

print("Orders duplicates:", Orders.duplicated().sum())
print("People duplicates:", People.duplicated().sum())
print("Returns duplicates:", Returns.duplicated().sum())

In [ ]:
Returns.shape

### 3.3.1 Remove Duplicate Return Records

In [ ]:
# Remove duplicate returned orders
returns_duplicates = DataCleaner(Returns)
returns_duplicates.remove_duplicates()

In [ ]:
Returns.shape

## 3.4 Clean Product Name Values

In [ ]:
# 3.4 Clean Product Name spaces

product_name = DataCleaner(Orders)
product_name.remove_extra_spaces("Product Name")

### 3.4.1 Validate Product Name Formatting

In [ ]:
# Check for leading, trailing, or extra spaces

mask = Orders["Product Name"].str.contains(r"^\s|\s$|\s{2,}",regex=True)
spaces = mask.sum()

logging.info(f"Whitespace issues found in Product Name: {spaces}")

## 3.5 Feature Engineering

In [ ]:
# 3.5 Feature Engineering

# 3.5.1 Shipping Duration:
Orders["Shipping Duration"] = (Orders["Ship Date"] - Orders["Order Date"]).dt.days

# 3.5.2 Profit Margin, Profit per Unit
Orders["Profit Margin"] = (Orders["Profit"] / Orders["Sales"]) * 100
Orders["Profit per Unit"] = Orders["Profit"] / Orders["Quantity"]

# 3.5.3 Sales Performance Category
Orders["Sales per Unit"] = Orders["Sales"] / Orders["Quantity"]
Orders["Sales Performance Category"] = pd.cut(Orders["Sales"], bins=[0, 100, 500, float("inf")], labels=["Low", "Medium", "High"])

### 3.5.1 Add Date Features

In [ ]:
class FeatureEngineer:
    def __init__(self, df):
        self.df = df
        
    # Extract useful date components
    def add_date_features(self, date_column, name):
        self.df[f"{name} Year"] = self.df[date_column].dt.year
        self.df[f"{name} Quarter"] = self.df[date_column].dt.quarter
        self.df[f"{name} Month"] = self.df[date_column].dt.month
        self.df[f"{name} Day"] = self.df[date_column].dt.day

        logging.info("Date features created successfully")

In [ ]:
date_engineer = FeatureEngineer(Orders)
date_engineer.add_date_features("Order Date" , "Order")
date_engineer.add_date_features("Ship Date" , "Ship")

In [ ]:
Orders.head(5)

# Phase 4: Exploratory Data Analysis (EDA)

## 4.1 Dataset Overview

In [ ]:
# Phase 4: Exploratory Data Analysis (EDA)

logging.info("Starting exploratory data analysis")

# 4.1 Dataset Overview
datasets = {"Orders": Orders, "People": People, "Returns": Returns}

# Display basic information for all datasets
for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 40)
    print("Shape:", df.shape)
    print("Rows:", len(df))
    print("Columns:", len(df.columns))

# Display data types for all datasets
for name, df in datasets.items():
    print(f"\n{name} Data Types")
    print("-" * 40)
    display(df.dtypes)

# Display key dataset counts
print("Orders:")
print("Unique Orders:", Orders["Order ID"].nunique())
print("Unique Customers:", Orders["Customer ID"].nunique())
print("Unique Products:", Orders["Product ID"].nunique())

print("\nPeople:")
print("People:", People["Person"].nunique())
print("Regions:", People["Region"].nunique())

print("\nReturns:")
print("Returned Orders:", Returns["Order ID"].nunique())

logging.info("Dataset overview completed successfully")

## 4.2 Descriptive Statistics

In [ ]:
# 4.2 Descriptive Statistics

logging.info("Starting descriptive statistics")
numeric_cols = ["Sales", "Profit", "Quantity", "Discount", "Shipping Duration", "Profit Margin", "Profit per Unit", "Sales per Unit"]

display(Orders[numeric_cols].describe().round(3))

### 4.2.1 Skewness and Kurtosis

In [ ]:
# Add skewness and kurtosis

stats = Orders[numeric_cols].describe().T

stats["skewness"] = Orders[numeric_cols].skew()
stats["kurtosis"] = Orders[numeric_cols].kurtosis()

display(stats)

### 4.2.2 Categorical Summary

In [ ]:
# Descriptive summary for categorical datasets
print("Orders:")
display(Orders.describe(include="string"))

print("People:")
display(People.describe(include="string"))

print("Returns:")
display(Returns.describe(include="string"))

## 4.3 Numerical Distributions

In [ ]:
# 4.3 Numerical Distributions

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].hist(Orders["Sales"], bins=100)
axes[0, 0].set_title("Sales Distribution")
axes[0, 0].set_xlabel("Sales")
axes[0, 0].set_ylabel("Frequency")

axes[0, 1].hist(Orders["Sales per Unit"], bins=100)
axes[0, 1].set_title("Sales per Unit Distribution")
axes[0, 1].set_xlabel("Sales per Unit")
axes[0, 1].set_ylabel("Frequency")

axes[1, 0].hist(Orders["Profit"], bins=100)
axes[1, 0].set_title("Profit Distribution")
axes[1, 0].set_xlabel("Profit")
axes[1, 0].set_ylabel("Frequency")

axes[1, 1].hist(Orders["Profit per Unit"], bins=100)
axes[1, 1].set_title("Profit per Unit Distribution")
axes[1, 1].set_xlabel("Profit per Unit")
axes[1, 1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

Sales             → extreme right tail
Sales per Unit    → extreme right tail
Profit            → extreme values
Profit per Unit   → extreme values

## 4.4 Outlier Analysis

### 4.4.1 Sales Boxplot

In [ ]:
# 4.4 Outlier Analysis

# 4.4.1 Sales Boxplot

plt.figure(figsize=(10, 5))

plt.boxplot(Orders["Sales"])

plt.ylabel("Sales")
plt.title("Sales Boxplot")

plt.show()

In [ ]:
# Calculate IQR for Sales

sales_q1 = Orders["Sales"].quantile(0.25)
sales_q3 = Orders["Sales"].quantile(0.75)

sales_iqr = sales_q3 - sales_q1

sales_lower_bound = sales_q1 - 1.5 * sales_iqr
sales_upper_bound = sales_q3 + 1.5 * sales_iqr

print("sales Q1:", sales_q1)
print("sales Q3:", sales_q3)
print("sales IQR:", sales_iqr)
print("sales Lower Bound:", sales_lower_bound)
print("sales Upper Bound:", sales_upper_bound)

# Calculate IQR for Sales per Unit

sales_per_unit_q1 = Orders["Sales per Unit"].quantile(0.25)
sales_per_unit_q3 = Orders["Sales per Unit"].quantile(0.75)

sales_per_unit_iqr = sales_per_unit_q3 - sales_per_unit_q1

sales_per_unit_lower_bound = sales_per_unit_q1 - 1.5 * sales_per_unit_iqr
sales_per_unit_upper_bound = sales_per_unit_q3 + 1.5 * sales_per_unit_iqr

print("\nsales per unit Q1:", sales_per_unit_q1)
print("sales per unit Q3:", sales_per_unit_q3)
print("sales per unit IQR:", sales_per_unit_iqr)
print("sales per unit Lower Bound:", sales_per_unit_lower_bound)
print("sales per unit Upper Bound:", sales_per_unit_upper_bound)

In [ ]:
# Count potential Sales outliers

sales_outliers = Orders[(Orders["Sales"] < sales_lower_bound) | (Orders["Sales"] > sales_upper_bound)]
sales_outliers_percent = (len(sales_outliers) / len(Orders)) * 100

print("Potential Sales outliers:", len(sales_outliers))
print("Percentage of Sales outliers:", sales_outliers_percent)

# Count potential Sales per Unit outliers

sales_per_unit_outliers = Orders[(Orders["Sales per Unit"] < sales_per_unit_lower_bound) | (Orders["Sales per Unit"] > sales_per_unit_upper_bound)]
sales_per_unit_outliers_percent = (len(sales_per_unit_outliers) / len(Orders)) * 100

print("Potential Sales per Unit outliers:", len(sales_per_unit_outliers))
print("Percentage of Sales per Unit outliers:", sales_per_unit_outliers_percent)

In [ ]:
# Inspect potential Sales outliers

display(sales_outliers[["Row ID","Order ID","Product Name","Sales","Quantity","Sales per Unit","Discount","Profit"]]
        .sort_values("Sales", ascending=False).head(20))

### 4.4.2 Profit Boxplot

In [ ]:
# 4.4.2 Profit Boxplot

plt.figure(figsize=(10, 5))

plt.boxplot(Orders["Profit"])

plt.ylabel("Profit")
plt.title("Profit Boxplot")

plt.show()

In [ ]:
# Calculate IQR for Profit

profit_q1 = Orders["Profit"].quantile(0.25)
profit_q3 = Orders["Profit"].quantile(0.75)
profit_iqr = profit_q3 - profit_q1
profit_lower_bound = profit_q1 - 1.5 * profit_iqr
profit_upper_bound = profit_q3 + 1.5 * profit_iqr

print("Profit Q1:", profit_q1)
print("Profit Q3:", profit_q3)
print("Profit IQR:", profit_iqr)
print("Profit Lower Bound:", profit_lower_bound)
print("Profit Upper Bound:", profit_upper_bound)

# Calculate IQR for Profit per Unit

profit_per_unit_q1 = Orders["Profit per Unit"].quantile(0.25)
profit_per_unit_q3 = Orders["Profit per Unit"].quantile(0.75)
profit_per_unit_iqr = profit_per_unit_q3 - profit_per_unit_q1
profit_per_unit_lower_bound = profit_per_unit_q1 - 1.5 * profit_per_unit_iqr
profit_per_unit_upper_bound = profit_per_unit_q3 + 1.5 * profit_per_unit_iqr

print("\nProfit per Unit Q1:", profit_per_unit_q1)
print("Profit per Unit Q3:", profit_per_unit_q3)
print("Profit per Unit IQR:", profit_per_unit_iqr)
print("Profit per Unit Lower Bound:", profit_per_unit_lower_bound)
print("Profit per Unit Upper Bound:", profit_per_unit_upper_bound)

In [ ]:
# Count potential Profit outliers

profit_outliers = Orders[(Orders["Profit"] < profit_lower_bound) | (Orders["Profit"] > profit_upper_bound)]

print("Potential Profit outliers:", len(profit_outliers))
print("Percentage of Profit outliers:", len(profit_outliers) / len(Orders) * 100)

# Count potential Profit per Unit outliers

profit_per_unit_outliers = Orders[(Orders["Profit per Unit"] < profit_per_unit_lower_bound) | (Orders["Profit per Unit"] > profit_per_unit_upper_bound)]

print("Potential Profit per Unit outliers:", len(profit_per_unit_outliers))
print("Percentage of Profit per Unit outliers:", len(profit_per_unit_outliers) / len(Orders) * 100)

In [ ]:
# Inspect potential Profit outliers

display(profit_outliers[["Row ID","Order ID","Product Name","Sales","Quantity","Discount","Profit","Profit Margin","Profit per Unit"]]
        .sort_values("Profit").head(20))
display(profit_outliers[["Row ID","Order ID","Product Name","Sales","Quantity","Discount","Profit","Profit Margin","Profit per Unit"]]
        .sort_values("Profit", ascending=False).head(20))

### 4.4.3 Profit Margin Distribution

In [ ]:
# 4.4.3 Profit Margin Distribution by Category

sns.catplot(data=Orders, x="Category", y="Profit Margin", kind="violin", aspect=1.5)

plt.title("Profit Margin Distribution by Category")
plt.xlabel("Category")
plt.ylabel("Profit Margin")

plt.show()

In [ ]:
# Calculate IQR for Profit Margin

pm_q1 = Orders["Profit Margin"].quantile(0.25)
pm_q3 = Orders["Profit Margin"].quantile(0.75)

pm_iqr = pm_q3 - pm_q1

pm_lower_bound = pm_q1 - 1.5 * pm_iqr
pm_upper_bound = pm_q3 + 1.5 * pm_iqr

print("Profit Margin Q1:", pm_q1)
print("Profit Margin Q3:", pm_q3)
print("Profit Margin IQR:", pm_iqr)
print("Profit Margin Lower Bound:", pm_lower_bound)
print("Profit Margin Upper Bound:", pm_upper_bound)

In [ ]:
# Count potential Profit Margin outliers

profit_margin_outliers = Orders[(Orders["Profit Margin"] < pm_lower_bound) |(Orders["Profit Margin"] > pm_upper_bound)]

print("Potential Profit Margin outliers:", len(profit_margin_outliers))
print("Percentage of Profit Margin outliers:", len(profit_margin_outliers) / len(Orders) * 100)

In [ ]:
# Inspect potential Profit Margin outliers

display(profit_margin_outliers[["Row ID", "Order ID", "Product Name", "Sales", "Quantity", "Discount", "Profit", "Profit Margin", "Profit per Unit"]]
        .sort_values("Profit Margin").head(20))

In [ ]:
Orders.columns

## 4.5 Categorical Performance Analysis

### 4.5.1 Sales Performance Across Key Dimensions


In [ ]:
# 4.5 Categorical Performance Analysis
# 4.5.1 Sales Performance Across Key Dimensions

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.barplot(x="Category", y="Sales", data=Orders, estimator=sum, errorbar=None, ax=axes[0, 0])
axes[0, 0].set_title("Sales by Category")
axes[0, 0].set_xlabel("Category")
axes[0, 0].set_ylabel("Total Sales")

sns.barplot(x="Segment", y="Sales", data=Orders, estimator=sum, errorbar=None, ax=axes[0, 1])
axes[0, 1].set_title("Sales by Customer Segment")
axes[0, 1].set_xlabel("Segment")
axes[0, 1].set_ylabel("Total Sales")

sns.barplot(x="Region", y="Sales", data=Orders, estimator=sum, errorbar=None, ax=axes[1, 0])
axes[1, 0].set_title("Sales by Region")
axes[1, 0].set_xlabel("Region")
axes[1, 0].set_ylabel("Total Sales")

sns.barplot(x="Ship Mode", y="Sales", data=Orders, estimator=sum, errorbar=None, ax=axes[1, 1])
axes[1, 1].set_title("Sales by Ship Mode")
axes[1, 1].set_xlabel("Ship Mode")
axes[1, 1].set_ylabel("Total Sales")

plt.tight_layout()
plt.show()

### 4.5.2 Profit Performance Across Key Dimensions

In [ ]:
# 4.5.2 Profit Performance Across Key Dimensions

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.barplot(x="Category", y="Profit", data=Orders, estimator=sum, errorbar=None, ax=axes[0, 0])
axes[0, 0].set_title("Profit by Category")
axes[0, 0].set_xlabel("Category")
axes[0, 0].set_ylabel("Total Profit")

sns.barplot(x="Segment", y="Profit", data=Orders, estimator=sum, errorbar=None, ax=axes[0, 1])
axes[0, 1].set_title("Profit by Customer Segment")
axes[0, 1].set_xlabel("Segment")
axes[0, 1].set_ylabel("Total Profit")

sns.barplot(x="Region", y="Profit", data=Orders, estimator=sum, errorbar=None, ax=axes[1, 0])
axes[1, 0].set_title("Profit by Region")
axes[1, 0].set_xlabel("Region")
axes[1, 0].set_ylabel("Total Profit")

sns.barplot(x="Ship Mode", y="Profit", data=Orders, estimator=sum, errorbar=None, ax=axes[1, 1])
axes[1, 1].set_title("Profit by Ship Mode")
axes[1, 1].set_xlabel("Ship Mode")
axes[1, 1].set_ylabel("Total Profit")

plt.tight_layout()
plt.show()

### 4.5.3 Categorical Performance Summary

In [ ]:
# 4.5 Categorical Performance Analysis

category_performance = Orders.groupby("Category").agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"))
category_performance["Profit Margin"] = (category_performance["Total_Profit"] / category_performance["Total_Sales"]) * 100

Region_performance = Orders.groupby("Region").agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"))
Region_performance["Profit Margin"] = (Region_performance["Total_Profit"] / Region_performance["Total_Sales"]) * 100

Segment_performance = Orders.groupby("Segment").agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"))
Segment_performance["Profit Margin"] = (Segment_performance["Total_Profit"] / Segment_performance["Total_Sales"]) * 100

Ship_Mode_performance = Orders.groupby("Ship Mode").agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"))
Ship_Mode_performance["Profit Margin"] = (Ship_Mode_performance["Total_Profit"] / Ship_Mode_performance["Total_Sales"]) * 100

display(category_performance)
display(Region_performance)
display(Segment_performance)
display(Ship_Mode_performance)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Category
category_performance.plot(kind="bar", ax=axes[0, 0])
axes[0, 0].set_title("Sales and Profit by Category")
axes[0, 0].set_xlabel("Category")
axes[0, 0].set_ylabel("Amount")
axes[0, 0].tick_params(axis="x", rotation=0)

# Segment
Segment_performance.plot(kind="bar", ax=axes[0, 1])
axes[0, 1].set_title("Sales and Profit by Segment")
axes[0, 1].set_xlabel("Segment")
axes[0, 1].set_ylabel("Amount")
axes[0, 1].tick_params(axis="x", rotation=0)

# Region
Region_performance.plot(kind="bar", ax=axes[1, 0])
axes[1, 0].set_title("Sales and Profit by Region")
axes[1, 0].set_xlabel("Region")
axes[1, 0].set_ylabel("Amount")
axes[1, 0].tick_params(axis="x", rotation=0)

# Ship Mode
Ship_Mode_performance.plot(kind="bar", ax=axes[1, 1])
axes[1, 1].set_title("Sales and Profit by Ship Mode")
axes[1, 1].set_xlabel("Ship Mode")
axes[1, 1].set_ylabel("Amount")
axes[1, 1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 4.6 Discount and Profitability Analysis

In [ ]:
# 4.6 Discount and Profitability Analysis

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.scatterplot(data=Orders, x="Discount", y="Profit", alpha=0.4, ax=axes[0])
axes[0].set_xlabel("Discount")
axes[0].set_ylabel("Profit")
axes[0].set_title("Discount vs Profit")

sns.scatterplot(data=Orders, x="Discount", y="Profit Margin", alpha=0.4, ax=axes[1])
axes[1].set_xlabel("Discount")
axes[1].set_ylabel("Profit Margin")
axes[1].set_title("Discount vs Profit Margin")

plt.tight_layout()
plt.show()

### 4.6.1 Discount and Profit Correlation

In [ ]:
# Calculate correlation between Discount and Profit

correlation = Orders["Discount"].corr(Orders["Profit Margin"])
print(f"Correlation between Discount and Profit Matgin: {correlation}")

correlation2 = Orders["Discount"].corr(Orders["Profit"])
print(f"Correlation between Discount and Profit: {correlation2}")

### 4.6.2 Correlation Matrix

In [ ]:
corr_matrix = Orders[["Sales", "Profit", "Quantity", "Discount", "Shipping Duration", "Profit Margin", "Profit per Unit", "Sales per Unit"]].corr()

# Visualize correlations between numeric variables

plt.figure(figsize=(10, 8))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",

    center=0
)

plt.title("Correlation Matrix")
plt.show()

In [ ]:
sns.pairplot(Orders[[ "Sales", "Discount", "Profit Margin", "Segment"]], hue="Segment")

plt.show()

## 4.7 Quantity vs Sales

In [ ]:
# 4.7 Quantity vs Sales

plt.figure(figsize=(10, 5))

sns.scatterplot(data=Orders, x="Quantity", y="Sales", alpha=0.4)

plt.xlabel("Quantity")
plt.ylabel("Sales")
plt.title("Quantity vs Sales")

plt.tight_layout()
plt.show()

### 4.7.1 Quantity and Sales Correlation

In [ ]:
# Calculate correlation between Quantity and Sales

correlation3 = Orders["Quantity"].corr(Orders["Sales"])
print(f"Correlation between Quantity and Sales: {correlation3}")

In [ ]:
Orders.columns

## 4.8 Sales and Profit Trends

In [ ]:
# 4.8.1 Year-over-Year Sales and Profit

yearly = (Orders.groupby("Order Year")[["Sales", "Profit"]].sum())
display(yearly)

### 4.8.1 Year-over-Year Sales and Profit

In [ ]:
# Yearly Sales and Profit Trends
sales = Orders.groupby("Order Year")["Sales"].sum()
Profit = Orders.groupby("Order Year")["Profit"].sum()

fig, axes = plt.subplots(2, 1, figsize=(10, 7))

# Yearly Sales
axes[0].plot(sales.index, sales.values, marker="o")
axes[0].set_title("Yearly Sales Trend")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Total Sales")

# Yearly Profit
axes[1].plot(Profit.index, Profit.values, marker="o", color="red")
axes[1].set_title("Yearly Profit Trend")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Total Profit")

plt.tight_layout()
plt.show()


### 4.8.2 Monthly Sales and Profit Trends

In [ ]:
# 4.8.2 Monthly Sales and Profit Trends

monthly = (Orders.groupby(Orders["Order Date"].dt.to_period("M"))[["Sales", "Profit"]].sum())

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Monthly Sales Trend
axes[0].plot(monthly.index.to_timestamp(),monthly["Sales"])

axes[0].set_title("Monthly Sales Trend")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Total Sales")

# Monthly Profit Trend
axes[1].plot(monthly.index.to_timestamp(), monthly["Profit"], color="red")

axes[1].set_title("Monthly Profit Trend")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Total Profit")

plt.tight_layout()
plt.show()

### 4.8.3 Monthly Sales Seasonality

In [ ]:
# 4.8.3 Monthly Sales Seasonality

monthly_seasonality = (Orders.groupby("Order Month")["Sales"].mean())

# Visualize monthly sales seasonality

plt.figure(figsize=(10, 5))

monthly_seasonality.plot(kind="bar")

plt.xlabel("Month")
plt.ylabel("Average Sales")
plt.title("Average Sales by Month")

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 4.8.4 Monthly Trend Visualization

In [ ]:
# 4.8.4 Monthly Sales and Profit Trends

monthly_sales = Orders.groupby("Order Month")["Sales"].mean()
monthly_profit = Orders.groupby("Order Month")["Profit"].mean()

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Monthly Sales Trend
axes[0].plot(monthly_sales.index, monthly_sales.values, marker="o")

axes[0].set_title("Monthly Sales Trend")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Total Sales")

# Monthly Profit Trend
axes[1].plot(monthly_profit.index, monthly_profit.values, marker="o", color="red")

axes[1].set_title("Monthly Profit Trend")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Total Profit")

plt.tight_layout()
plt.show()

## 4.9 Customer Performance Analysis

In [ ]:
# 4.9.1 Customer Performance

customer_performance = (Orders.groupby(["Customer ID", "Customer Name"]).agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"))
                        .sort_values("Total_Sales", ascending=False))

# 4.9.2 Customer Profitability
customer_profitability = (Orders.groupby(["Customer ID", "Customer Name"]).agg(Total_Sales=("Sales", "sum"),Total_Profit=("Profit", "sum"))
                        .sort_values("Total_Profit", ascending=False))

display(customer_profitability.head(10))
display(customer_performance.head(10))

### 4.9.1 Top Customers by Sales and Profit

In [ ]:
# Top 10 customers by total sales and profit
top_customers = customer_performance.head(10)
top_profit_customers = customer_profitability.head(10)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Top 10 Customers by Sales
sns.barplot(x=top_customers["Total_Sales"], y=top_customers.index.get_level_values("Customer Name"), errorbar=None, ax=axes[0])

axes[0].set_xlabel("Total Sales")
axes[0].set_ylabel("Customer")
axes[0].set_title("Top 10 Customers by Sales")


# Top 10 Customers by Profit
sns.barplot(x=top_profit_customers["Total_Profit"], y=top_profit_customers.index.get_level_values("Customer Name"), errorbar=None, ax=axes[1])

axes[1].set_xlabel("Total Profit")
axes[1].set_ylabel("Customer")
axes[1].set_title("Top 10 Customers by Profit")

plt.tight_layout()
plt.show()

## 4.10 People Analysis

In [ ]:
# 4.10.1 Merge People with Orders

orders_people = pd.merge(Orders, People, on="Region", how="left")

display(orders_people[["Person", "Region", "Sales", "Profit"]].head())

### 4.10.1 Sales and Profit by Person

In [ ]:
# 4.10.2 Sales and Profit by Person

person_performance = (orders_people.groupby("Person").agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"))
                      .sort_values("Total_Sales", ascending=False))

display(person_performance)

### 4.10.2 People Performance Visualization

In [ ]:
# Visualize sales and profit by person

person_performance.plot(kind="bar", figsize=(10, 5))

plt.xlabel("Person")
plt.ylabel("Amount")
plt.title("Sales and Profit by Person")
plt.xticks(rotation=0)
plt.show()

## 4.11 Returns Analysis

In [ ]:
# 4.11.1 Merge Returns with Orders

orders_returns = pd.merge(Orders, Returns, on="Order ID", how="left")
orders_returns["Returned"] = (orders_returns["Returned"].fillna("No"))

display(orders_returns[["Order ID", "Returned", "Sales", "Profit"]].head())

In [ ]:
returned_counts = orders_returns["Returned"].value_counts(dropna=False)
returned_percentage = returned_counts / returned_counts.sum() * 100

display(returned_percentage)

### 4.11.1 Return Rate

In [ ]:
# 4.11.1 Return Rate
return_rate = (orders_returns[orders_returns["Returned"] == "Yes"]["Order ID"].nunique() / orders_returns["Order ID"].nunique()) * 100
print(f"Return Rate: {return_rate}")

### 4.11.2 Returned vs Non-returned Performance

In [ ]:
# 4.12.2 Returned vs Non-returned Performance
return_performance = (orders_returns.groupby("Returned").agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum")).round(2))

display(return_performance)

### 4.11.3 Sales and Profit by Return Status

In [ ]:
# Compare Sales and Profit by Return Status

return_performance[["Total_Sales", "Total_Profit"]].plot(kind="bar", figsize=(10, 5))

plt.xlabel("Return Status")
plt.ylabel("Amount")
plt.title("Sales and Profit by Return Status")
plt.xticks(rotation=0)
plt.show()

### 4.11.4 Return Rate Across Key Dimensions

In [ ]:
category_returns = orders_returns.groupby("Category")["Order ID"].nunique()
returned_category = orders_returns[orders_returns["Returned"] == "Yes"].groupby("Category")["Order ID"].nunique()
category_returns = returned_category / category_returns * 100

region_returns = orders_returns.groupby("Region")["Order ID"].nunique()
returned_region = orders_returns[orders_returns["Returned"] == "Yes"].groupby("Region")["Order ID"].nunique()
region_returns = returned_region / region_returns * 100

segment_returns = orders_returns.groupby("Segment")["Order ID"].nunique()
returned_segment = orders_returns[orders_returns["Returned"] == "Yes"].groupby("Segment")["Order ID"].nunique()
segment_returns = returned_segment / segment_returns * 100

ship_returns = orders_returns.groupby("Ship Mode")["Order ID"].nunique()
returned_ship = orders_returns[orders_returns["Returned"] == "Yes"].groupby("Ship Mode")["Order ID"].nunique()
ship_returns = returned_ship / ship_returns * 100

# 4.11.4 Returns Analysis

fig, axes = plt.subplots(2, 2, figsize=(15, 8))

# Return Rate by Category
category_returns.plot(kind="bar", ax=axes[0, 0])
axes[0, 0].set_title("Return Rate by Category")
axes[0, 0].set_xlabel("Category")
axes[0, 0].set_ylabel("Return Rate (%)")
axes[0, 0].tick_params(axis="x", rotation=0)

# Return Rate by Region
region_returns.plot(kind="bar", ax=axes[0, 1])
axes[0, 1].set_title("Return Rate by Region")
axes[0, 1].set_xlabel("Region")
axes[0, 1].set_ylabel("Return Rate (%)")
axes[0, 1].tick_params(axis="x", rotation=0)

# Return Rate by Segment
segment_returns.plot(kind="bar", ax=axes[1, 0])
axes[1, 0].set_title("Return Rate by Segment")
axes[1, 0].set_xlabel("Segment")
axes[1, 0].set_ylabel("Return Rate (%)")
axes[1, 0].tick_params(axis="x", rotation=0)

# Return Rate by Ship Mode
ship_returns.plot(kind="bar", ax=axes[1, 1])
axes[1, 1].set_title("Return Rate by Ship Mode")
axes[1, 1].set_xlabel("Ship Mode")
axes[1, 1].set_ylabel("Return Rate (%)")
axes[1, 1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 4.12 Final EDA Summary

In [ ]:
# 4.12 Final EDA Summary

logging.info("Generating final EDA summary")

print("=" * 60)
print("FINAL EDA SUMMARY")
print("=" * 60)

# 1. Key Business KPIs
print("\nKPI Summary:")
print("-" * 40)

total_sales = Orders["Sales"].sum()
total_profit = Orders["Profit"].sum()
overall_margin = total_profit / total_sales * 100

print(f"Total Sales: {total_sales:,.2f}")
print(f"Total Profit: {total_profit:,.2f}")
print(f"Overall Profit Margin: {overall_margin:.2f}%")
print(f"Unique Orders: {Orders['Order ID'].nunique():}")
print(f"Unique Customers: {Orders['Customer ID'].nunique():}")
print(f"Unique Products: {Orders['Product ID'].nunique():}")
print(f"Return Rate: {return_rate:.2f}")

# 2. Top-performing dimensions
print("\nTop Performers:")
print("-" * 40)
print("Top Category:",Orders.groupby("Category")["Sales"].sum().idxmax())
print("Top Segment:",Orders.groupby("Segment")["Sales"].sum().idxmax())
print("Top Region:",Orders.groupby("Region")["Sales"].sum().idxmax())

# 3. Key relationships
print("\nKey Relationships:")
print("-" * 40)
discount_margin_corr = Orders["Discount"].corr(Orders["Profit Margin"])
quantity_sales_corr = Orders["Quantity"].corr(Orders["Sales"])
print(f"Discount vs Profit Margin: {discount_margin_corr:.3f}")
print(f"Quantity vs Sales: {quantity_sales_corr:.3f}")

# 4. Outlier summary
print("\nOutlier Summary:")
print("-" * 40)
print(f"Sales potential outliers: {len(sales_outliers)}")
print(f"Profit potential outliers: {len(profit_outliers)}")
print(f"Profit Margin potential outliers: "f"{len(profit_margin_outliers)}")
print("\nOutlier Treatment Decision:")
print("Potential outliers were retained because they")
print("appeared to represent legitimate business transactions.")

# 5. Returns
print("\nReturns:")
print("-" * 40)
print(f"Return Rate: {return_rate:.2f}%")

logging.info("Final EDA summary completed successfully")

# Phase 5: Diagnostic Analysis

In [ ]:
Orders.columns

## 5.1 Discount Impact on Profitability

In [ ]:
# 5.1 Discount Impact on Profitability

discount_analysis = (Orders.groupby("Discount").agg(Average_Profit=("Profit", "mean"), Total_Profit=("Profit", "sum"), Total_Sales=("Sales", "sum"), Count=("Row ID", "count")))
discount_analysis["Profit Margin"] = (discount_analysis["Total_Profit"] / discount_analysis["Total_Sales"]) * 100

display(discount_analysis)
# Interpretation: Profitability declines noticeably as the discount level increases.

## 5.2 Loss Drivers

### 5.2.1 Loss-making Transactions

In [ ]:
# 5.2.1 Loss-making Transactions

loss_orders = Orders[Orders["Profit"] < 0]
total_losses = loss_orders["Profit"].sum()

print("Loss transactions:", len(loss_orders))
print("Percentage of loss-making transactions:", len(loss_orders) / len(Orders) * 100)
print(f"Total losses: {total_losses}")

### 5.2.2 Losses by Sub-Category

In [ ]:
# 5.2.2 Losses by Sub-Category

loss_by_subcategory = (loss_orders.groupby("Sub-Category").agg(Loss_Amount=("Profit", lambda x: -x.sum()), Loss_Transactions=("Order ID", "count"))
                       .sort_values("Loss_Amount", ascending=False))

display(loss_by_subcategory)

### 5.2.3 Largest Loss Drivers Visualization

In [ ]:
# Visualize largest loss drivers

loss_by_subcategory.head(10)["Loss_Amount"].sort_values().plot(kind="barh", figsize=(10, 5))

plt.xlabel("Total Loss")
plt.ylabel("Sub-Category")
plt.title("Top 10 Sub-Categories by Loss")
plt.tight_layout()
plt.show()

# Binders   → 40000
# Tables    → 33000
# Machines  → 30000

### 5.2.4 Losses by Discount Level

In [ ]:
# 5.2.3 Losses by Discount Level

loss_by_discount = (loss_orders.groupby("Discount").agg(Total_Loss=("Profit", lambda x: -x.sum()),Loss_Transactions=("Order ID", "count"))
                    .sort_values("Total_Loss", ascending=False))

display(loss_by_discount)
# High discount levels are associated with substantial losses. Discounts of 70% and 80% recorded the highest total losses

## 5.3 Sub-Category Profitability Drivers

In [ ]:
# 5.3 Sub-Category Profitability Drivers

subcategory_analysis = (Orders.groupby("Sub-Category").agg(Total_Sales=("Sales", "sum"),Total_Profit=("Profit", "sum"),Average_Discount=("Discount", "mean")))
subcategory_analysis["Profit Margin"] = (subcategory_analysis["Total_Profit"] / subcategory_analysis["Total_Sales"]) * 100

display(subcategory_analysis.sort_values("Profit Margin", ascending=False))
# Labels / Paper / Envelopes → Highest Profit Margins 
# Tables / Bookcases / Supplies → Negative Profit Margins 
# Higher Discount → Lower Profitability → Higher Losses

### 5.3.1 Loss Drivers by Product

In [ ]:
# 5.3 Loss Drivers by Product

product_losses = (Orders[Orders["Profit"] < 0].groupby(["Product Name", "Sub-Category"]).agg(
    Total_Profit=("Profit", "sum"),Total_Sales=("Sales", "sum"),Avg_Discount=("Discount", "mean"),Loss_Transactions=("Profit", "count")))

product_losses["Profit Margin"] = (product_losses["Total_Profit"] / product_losses["Total_Sales"]) * 100
display(product_losses.sort_values("Total_Profit").head(10))

## 5.4 Shipping Performance

In [ ]:
# 5.4 Shipping Performance

shipping_analysis = (Orders.groupby("Ship Mode").agg(Average_Shipping_Duration=("Shipping Duration", "mean"), Total_Profit=("Profit", "sum"),Total_Sales=("Sales", "sum")))
shipping_analysis["Profit Margin"] = (shipping_analysis["Total_Profit"] / shipping_analysis["Total_Sales"]) * 100

display(shipping_analysis.sort_values("Average_Shipping_Duration"))
# Shipping performance shows limited variation in profit margin across ship modes.
# Standard Class has the longest average duration, but profit margins remain relatively similar.
# Shipping duration does not appear to be a major driver of profitability.

## 5.5 People Performance

In [ ]:
# 5.5 People Performance

people_performance = orders_people.groupby(["Person", "Region"]).agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"))
people_performance["Profit Margin"] = (people_performance["Total_Profit"] / people_performance["Total_Sales"]) * 100

display(people_performance.sort_values("Profit Margin", ascending=False))

# Phase 6: Prescriptive Analysis

## 6.1 Discount Recommendations

In [ ]:
# Identify profitable and loss-making discount levels

profitable_discounts = discount_analysis[discount_analysis["Profit Margin"] > 0].index.tolist()
loss_making_discounts = discount_analysis[discount_analysis["Profit Margin"] < 0].index.tolist()

print("Profitable discount levels:", profitable_discounts)
print("Loss-making discount levels:", loss_making_discounts)

## 6.2 Product and Sub-Category Recommendations

In [ ]:
# 6.2 Product / Sub-Category Recommendations

high_profit_subcategories = subcategory_analysis.sort_values("Profit Margin", ascending=False).head(3)
loss_making_subcategories = subcategory_analysis.sort_values("Profit Margin").head(3)

display(high_profit_subcategories)
display(loss_making_subcategories)
# High-margin sub-categories should be prioritized for growth.
# while loss-making sub-categories should be reviewed for pricing and discount policies.

## 6.3 Returns Recommendations

In [ ]:
# 6.3 Returns Recommendations

print("Highest Category:", category_returns.idxmax(), category_returns.max())
print("Highest Region:", region_returns.idxmax(), region_returns.max())
print("Highest Segment:", segment_returns.idxmax(), segment_returns.max())
print("Highest Ship Mode:", ship_returns.idxmax(), ship_returns.max())
# Focus return reduction efforts on the areas with the highest return rates.

### 7.2 Final Business Recommendations

1. Discount
Keep discounts at lower levels where possible, especially avoid aggressive discounts of 30%+.

2. Products
Prioritize high-margin sub-categories such as Labels, Paper, and Envelopes.
Review loss-making sub-categories such as Tables, Bookcases, and Supplies.

3. Shipping
Shipping duration showed limited impact on profitability.
No major profitability-related changes are recommended.

4. Returns
Focus return-reduction efforts on areas with the highest return rates.

# Phase 7: Performance Optimization

## 7.1 Memory Usage Assessment


In [ ]:
# 7.1 Memory Usage Assessment

print("Orders Memory Usage:")
print("-" * 40)
Orders.info(memory_usage="deep")

In [ ]:
print("\nPeople Memory Usage:")
print("-" * 40)
People.info(memory_usage="deep")

In [ ]:
print("\nReturns Memory Usage:")
print("-" * 40)
Returns.info(memory_usage="deep")

## 7.2 Drop Unnecessary Columns

In [ ]:
Orders.columns

In [ ]:
# 7.2 Drop Unnecessary Columns

logging.info("Removing unnecessary columns for export")

Orders = Orders.drop(columns=[
    "Order Year",
    "Order Quarter",
    "Order Month",
    "Order Day",
    "Ship Year",
    "Ship Quarter",
    "Ship Month",
    "Ship Day"
])

logging.info("Unnecessary columns removed successfully")

## 7.3 Optimize Data Types


In [ ]:
logging.info("Optimizing categorical data types")

Orders["Ship Mode"] = Orders["Ship Mode"].astype("category")
Orders["Segment"] = Orders["Segment"].astype("category")
Orders["Region"] = Orders["Region"].astype("category")
Orders["Category"] = Orders["Category"].astype("category")
Orders["Sub-Category"] = Orders["Sub-Category"].astype("category")

logging.info("Categorical data types optimized successfully")

## 7.4 Downcast Numeric Data

In [ ]:
# 7.4 Downcast Numeric Data

logging.info("Optimizing numeric data types")

Orders["Row ID"] = Orders["Row ID"].astype("int32")
Orders["Quantity"] = Orders["Quantity"].astype("int8")

Orders["Sales"] = Orders["Sales"].astype("float32")
Orders["Discount"] = Orders["Discount"].astype("float32")
Orders["Profit"] = Orders["Profit"].astype("float32")
Orders["Profit Margin"] = Orders["Profit Margin"].astype("float32")
Orders["Profit per Unit"] = Orders["Profit per Unit"].astype("float32")
Orders["Sales per Unit"] = Orders["Sales per Unit"].astype("float32")

logging.info("Numeric data types optimized successfully")

## 7.5 Memory After

In [ ]:
# 7.5 Memory After
Orders.info(memory_usage=True)

In [ ]:
memory_before = 9.3
memory_after = 1.2

Memory_Reduction = (memory_before - memory_after)
reduction_percentage = Memory_Reduction / memory_before * 100

print(f"Memory Reduction: {Memory_Reduction:.1f}")
print(f"Memory Reduction percentage: {reduction_percentage:.1f}%")

# Phase 8: Reporting and Export

## 8.1 KPI Summary

In [ ]:
# 8.1 KPI Summary

logging.info("Generating KPI summary")

kpi_summary = (
    f"Total Sales: {total_sales:,.2f}\n"
    f"Total Profit: {total_profit:,.2f}\n"
    f"Overall Profit Margin: {overall_margin:.2f}%\n"
    f"Unique Orders: {Orders['Order ID'].nunique()}\n"
    f"Unique Customers: {Orders['Customer ID'].nunique()}\n"
    f"Unique Products: {Orders['Product ID'].nunique()}\n"
    f"Return Rate: {return_rate:.2f}%"
)

print("KPI Summary:")
print("-" * 40)
print(kpi_summary)

logging.info("KPI summary generated successfully")

## 8.2 Analytical Report


In [ ]:
# 8.2 Analytical Report

logging.info("Generating analytical report")

report = (
    f"Top Category: {Orders.groupby('Category')['Sales'].sum().idxmax()}\n"
    f"Top Segment: {Orders.groupby('Segment')['Sales'].sum().idxmax()}\n"
    f"Top Region: {Orders.groupby('Region')['Sales'].sum().idxmax()}\n"
    f"Discount Impact: Discount vs Profit Margin correlation = {discount_margin_corr:.3f}\n"
    f"Quantity and Sales: Quantity vs Sales correlation = {quantity_sales_corr:.3f}\n"
    f"Main Loss Driver: {loss_by_subcategory.index[0]}\n"
    f"Highest Profit Margin: {subcategory_analysis['Profit Margin'].idxmax()}\n"
    f"Returns: {category_returns.idxmax()} has the highest return rate by category"
)

print("Analytical Report:")
print("-" * 40)
print(report)

logging.info("Analytical report generated successfully")

In [ ]:
with open("analytical_report.txt", "w", encoding="utf-8") as file:
    file.write(kpi_summary + "\n")
    file.write("-" * 40 + "\n")
    file.write(report)

## 8.3 Cleaned Data Export


In [ ]:
# 8.3 Cleaned Data Export

logging.info("Exporting cleaned datasets")

with pd.ExcelWriter("cleaned Sample - Superstore 2019.xlsx") as writer:
    Orders.to_excel(writer, sheet_name="Orders", index=False)
    People.to_excel(writer, sheet_name="People", index=False)
    Returns.to_excel(writer, sheet_name="Returns", index=False)

logging.info("Cleaned datasets exported successfully")

In [ ]:
# 8.2 Analytical Report

logging.info("Generating analytical report")

report = {
    "Top Category": Orders.groupby("Category")["Sales"].sum().idxmax(),
    "Top Segment": Orders.groupby("Segment")["Sales"].sum().idxmax(),
    "Top Region": Orders.groupby("Region")["Sales"].sum().idxmax(),
    "Discount Impact": f"Discount vs Profit Margin correlation = {discount_margin_corr:.3f}",
    "Quantity and Sales": f"Quantity vs Sales correlation = {quantity_sales_corr:.3f}",
    "Main Loss Driver": loss_by_subcategory.index[0],
    "Highest Profit Margin": subcategory_analysis["Profit Margin"].idxmax(),
    "Returns": f"{category_returns.idxmax()} has the highest return rate by category"
}

print("Analytical Report:")
print("-" * 40)

for key, value in report.items():
    print(f"{key}: {value}")

logging.info("Analytical report generated successfully")

# Final Project Summary
This project covered data loading, data quality assessment, cleaning, feature engineering, EDA, diagnostic analysis, and prescriptive analysis.

### Key Findings
- Higher discounts are strongly associated with lower profit margins.
- High discount levels are linked to substantial losses.
- Tables, Bookcases, and Supplies show negative overall profit margins.
- Labels, Paper, and Envelopes have the highest profit margins.
- Shipping duration does not appear to be a major driver of profitability.
- Return rates vary across the main business dimensions.

### Key Recommendations
- Avoid aggressive discounting, especially at 30% or higher.
- Prioritize high-margin sub-categories and review loss-making products.
- Maintain the current shipping approach unless other operational issues are identified.
- Investigate areas with relatively high return rates.